# 01 - Data Preprocessing (FD001)

**Purpose:** Prepare NASA C-MAPSS FD001 data for LSTM-based RUL prediction.

**Inputs:** `data/raw/train_FD001.txt`, `data/raw/test_FD001.txt`, `data/raw/RUL_FD001.txt`

**Outputs:** processed arrays and metadata in `data/processed/` for model training, evaluation, and inference demo.

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 30
RUL_CAP = 125

In [ ]:
columns = ["unit_id", "cycle", "op_setting_1", "op_setting_2", "op_setting_3"] + [f"sensor_{i}" for i in range(1, 22)]

train_df = pd.read_csv(RAW_DIR / "train_FD001.txt", sep=r"\s+", header=None, names=columns)
test_df = pd.read_csv(RAW_DIR / "test_FD001.txt", sep=r"\s+", header=None, names=columns)
rul_df = pd.read_csv(RAW_DIR / "RUL_FD001.txt", sep=r"\s+", header=None, names=["final_rul"])

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("RUL shape:", rul_df.shape)
train_df.head()

In [ ]:
drop_cols = [
    "op_setting_3",
    "sensor_1", "sensor_5", "sensor_6", "sensor_10",
    "sensor_16", "sensor_18", "sensor_19"
]

train_df = train_df.drop(columns=drop_cols).copy()
test_df = test_df.drop(columns=drop_cols).copy()

sensor_feature_cols = [c for c in train_df.columns if c.startswith("sensor_")]
print("Sensor feature count (model input):", len(sensor_feature_cols))
print(sensor_feature_cols)

In [ ]:
max_cycle = train_df.groupby("unit_id")["cycle"].max().rename("max_cycle")
train_df = train_df.join(max_cycle, on="unit_id")
train_df["rul_raw"] = train_df["max_cycle"] - train_df["cycle"]
train_df["rul"] = train_df["rul_raw"].clip(upper=RUL_CAP)
train_df = train_df.drop(columns=["max_cycle"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(train_df["rul_raw"], bins=50, color="#2563eb", alpha=0.85)
axes[0].set_title("RUL Distribution Before Clipping")
axes[0].set_xlabel("RUL")
axes[0].set_ylabel("Count")

axes[1].hist(train_df["rul"], bins=50, color="#f59e0b", alpha=0.85)
axes[1].set_title("RUL Distribution After Clipping (cap=125)")
axes[1].set_xlabel("RUL")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
def normalize_per_engine(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    # Per-engine scaling keeps each engine trajectory in a comparable 0-1 range.
    out = df.copy()
    out[feature_cols] = out[feature_cols].astype(float)
    for unit_id, idx in out.groupby("unit_id").groups.items():
        scaler = MinMaxScaler()
        out.loc[idx, feature_cols] = scaler.fit_transform(out.loc[idx, feature_cols])
    return out

train_df = normalize_per_engine(train_df, sensor_feature_cols)
test_df = normalize_per_engine(test_df, sensor_feature_cols)

train_df[sensor_feature_cols].describe().T.head()

In [ ]:
def build_train_sequences(df: pd.DataFrame, feature_cols: list[str], seq_len: int):
    X, y, seq_unit_ids = [], [], []
    for unit_id, grp in df.groupby("unit_id"):
        grp = grp.sort_values("cycle")
        feats = grp[feature_cols].to_numpy(dtype=np.float32)
        labels = grp["rul"].to_numpy(dtype=np.float32)

        for i in range(len(grp) - seq_len + 1):
            X.append(feats[i:i + seq_len])
            y.append(labels[i + seq_len - 1])
            seq_unit_ids.append(unit_id)

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32), np.asarray(seq_unit_ids, dtype=np.int32)


def build_test_last_sequences(test_data: pd.DataFrame, rul_data: pd.DataFrame, feature_cols: list[str], seq_len: int):
    X_test, y_test, unit_ids = [], [], []
    final_rul_map = {i + 1: v for i, v in enumerate(rul_data["final_rul"].tolist())}

    for unit_id, grp in test_data.groupby("unit_id"):
        grp = grp.sort_values("cycle")
        feats = grp[feature_cols].to_numpy(dtype=np.float32)

        if len(feats) < seq_len:
            pad = np.repeat(feats[[0]], seq_len - len(feats), axis=0)
            feats = np.vstack([pad, feats])

        X_test.append(feats[-seq_len:])
        y_test.append(float(final_rul_map[unit_id]))
        unit_ids.append(unit_id)

    return np.asarray(X_test, dtype=np.float32), np.asarray(y_test, dtype=np.float32), np.asarray(unit_ids, dtype=np.int32)

In [ ]:
X_train, y_train, train_unit_ids = build_train_sequences(train_df, sensor_feature_cols, SEQ_LEN)
X_test, y_test, test_unit_ids = build_test_last_sequences(test_df, rul_df, sensor_feature_cols, SEQ_LEN)

np.save(PROCESSED_DIR / "X_train_sequences.npy", X_train)
np.save(PROCESSED_DIR / "y_train_sequences.npy", y_train)
np.save(PROCESSED_DIR / "train_sequence_unit_ids.npy", train_unit_ids)
np.save(PROCESSED_DIR / "X_test_last.npy", X_test)
np.save(PROCESSED_DIR / "y_test_last.npy", y_test)
np.save(PROCESSED_DIR / "test_unit_ids.npy", test_unit_ids)

with open(PROCESSED_DIR / "feature_columns.json", "w", encoding="utf-8") as fp:
    json.dump(sensor_feature_cols, fp, indent=2)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)
print("Processed arrays saved to:", PROCESSED_DIR)

In [ ]:
final_rul_map = {i + 1: v for i, v in enumerate(rul_df["final_rul"].tolist())}
test_trajectories = {}

for unit_id, grp in test_df.groupby("unit_id"):
    grp = grp.sort_values("cycle").copy()
    max_cycle_obs = int(grp["cycle"].max())
    grp["actual_rul"] = (max_cycle_obs - grp["cycle"]) + final_rul_map[unit_id]

    seqs, actuals, cycles = [], [], []
    feats = grp[sensor_feature_cols].to_numpy(dtype=np.float32)
    act = grp["actual_rul"].to_numpy(dtype=np.float32)
    cyc = grp["cycle"].to_numpy(dtype=np.int32)

    for i in range(len(grp)):
        window = feats[max(0, i - SEQ_LEN + 1): i + 1]
        if len(window) < SEQ_LEN:
            pad = np.repeat(window[[0]], SEQ_LEN - len(window), axis=0)
            window = np.vstack([pad, window])

        seqs.append(window.astype(np.float32))
        actuals.append(float(act[i]))
        cycles.append(int(cyc[i]))

    test_trajectories[int(unit_id)] = {
        "cycles": cycles,
        "actual_rul": actuals,
        "sequences": np.asarray(seqs, dtype=np.float32),
        "final_rul": float(final_rul_map[unit_id])
    }

with open(PROCESSED_DIR / "test_engine_trajectories.pkl", "wb") as fp:
    pickle.dump(test_trajectories, fp)

sample_count = min(80, len(X_test))
sample_json = {"samples": X_test[:sample_count].tolist()}
with open(PROCESSED_DIR / "test_sequences_sample.json", "w", encoding="utf-8") as fp:
    json.dump(sample_json, fp)

print("Saved trajectory file and frontend sample JSON.")

In [ ]:
corr = train_df[sensor_feature_cols].corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Sensor Correlation Heatmap (FD001, Processed Features)")
plt.tight_layout()
plt.show()

### Done
Processed data is now ready in `data/processed/` for Notebook 02 (training).